In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import os
import sys
import time

# fix an issue with the method of running in the notebook
# I probably just need to look into python packaging more
sys.path.append(os.path.join(os.getcwd(), "scripts"))

import numpy as np

import tensorflow as tf
from tensorflow.keras import Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

from scripts.utils import Config
from scripts.utils.tf.dataloaders import AlmLoader
from scripts.utils.tf.plots import plot_histogram, plot_metrics, plot_predictions
from scripts.utils.tf.callbacks import TimedLoggingCallback, WarmupLearningRate

from scripts.attn_alm import alm_model

2024-02-28 18:45:47.860771: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-02-28 18:45:47.860835: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-02-28 18:45:47.862203: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-02-28 18:45:47.870623: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-02-28 18:45:49.104429: W tensorflow/compiler/tf2

In [3]:
s = Config(["settings/l1000_n512.json", "--nsims", "200", "--narray", "500", "--disable_noise"])

MAX_EPOCHS = 300
BATCH_SIZE = 64

# just some info for the model name
timestamp = int(time.time())
model_settings = {
    "dropout_rate": 0.3,
    "name": f"tester",
}

data_loader_args = {
    "shuffle": True,
    "seed": None,
    "batch_size": BATCH_SIZE,
    "cache": True,
    "shuffle_buffer": 1000,
}

# additional metrics we are intrested in
metrics = ["mean_absolute_error"]

In [6]:
lr_schedule = WarmupLearningRate(
    warmup_learning_rate=1e-8,  # start small
    warmup_steps=1e6,
    warmup_scale=1.5,
    warmup_scale_steps=1,
    warmed_learning_rate=1e-3,
    decay_steps=10000,
    decay_rate=0.95,
    staircase=True,
)

WarmupLearningRate: Warmup Range: 1e-08 -> 0.015000010000000001


In [7]:
strategy = tf.distribute.MirroredStrategy()
print(strategy.num_replicas_in_sync)
# strategy = tf.distribute.OneDeviceStrategy(device="/CPU:0")
num_gpus = strategy.num_replicas_in_sync

data_loader = AlmLoader(
    s.alm_file, num_replicas=num_gpus, **data_loader_args
)
train_dataset, test_dataset, val_dataset = data_loader.get_split(0.8, 0.1, 0.1)

with strategy.scope():
    opt = Adam(learning_rate=lr_schedule)

    model = alm_model(Input(data_loader.shape), **model_settings)
    # model = simple_transformer(Input(data_loader.shape), **model_settings)

    model.compile(optimizer=opt, loss=tf.keras.losses.mse, metrics=metrics)

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0',)
1


In [8]:
model.summary()

Model: "tester"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 2, 1000, 1000)]      0         []                            
                                                                                                  
 dense (Dense)               (None, 2, 1000, 512)         512512    ['input_1[0][0]']             
                                                                                                  
 dense_1 (Dense)             (None, 2, 1000, 128)         65664     ['dense[0][0]']               
                                                                                                  
 dense_2 (Dense)             (None, 2, 1000, 1)           129       ['dense_1[0][0]']             
                                                                                             

In [9]:
callbacks = [
    # We use earlystoping to prevent overfitting
    EarlyStopping(
        monitor="val_loss",
        patience=20,
        verbose=1,
        restore_best_weights=True,
        start_from_epoch=50,
    ),
    # TimedLoggingCallback(print_frequency=60),
    # TensorBoard(
    #     log_dir=f"{s.tb_dir}/{model_settings['name']}",
    #     histogram_freq=1,
    # ),
]

In [10]:
if False:
    import wandb
    from wandb.keras import WandbMetricsLogger

    # wandb.tensorboard.patch(root_logdir=s.tb_dir)

    wandb.init(
        project="mlpng",
        tags=["tester", "dev"],
        config=s.settings | model_settings,
        dir="data",
        sync_tensorboard=True,
    )

    # Add the wandb logger to the callbacks, so it is used
    callbacks.append(WandbMetricsLogger())

In [11]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

Epoch 1/300


2024-02-28 18:47:47.017444: W tensorflow/core/grappler/optimizers/data/auto_shard.cc:553] The `assert_cardinality` transformation is currently not handled by the auto-shard rewrite and will be removed.


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Redu

2024-02-28 18:47:58.133059: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] ShuffleDatasetV3:4: Filling up shuffle buffer (this may take a while): 589 of 1000
2024-02-28 18:48:05.072138: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:452] Shuffle buffer filled.
2024-02-28 18:48:09.515849: I external/local_xla/xla/service/service.cc:168] XLA service 0x154bbabf4b10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-02-28 18:48:09.515885: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA A100-SXM4-80GB, Compute Capability 8.0
2024-02-28 18:48:09.522188: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-02-28 18:48:09.914318: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
I0000 00:00:1709167690.005042 4110076 device_compiler.h:186] Compiled cluste

 514/1250 [===========>..................] - ETA: 30:19 - loss: 333244.5938 - mean_absolute_error: 500.5809

: 

: 

: 

In [ ]:
# Lets plot the predictions from the unseen test set
y_pred = model.predict(test_dataset, verbose=0).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_dataset])

In [ ]:
# Plot the loss curves and metrics
plot_metrics(history, metrics=["loss"] + metrics)
plot_predictions(y_test, y_pred)
plot_histogram(y_test, y_pred)